In [ ]:
import os
import random
import math
import shutil
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torchvision import datasets, transforms, models
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    classification_report,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Fixação de semente para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
print(f"Treinando em: {device}")

DATA_DIR = "../dataset/preprocess_imgs"
MODELS_DIR = "../models"
RUNS_DIR = "../runs/ViT_run"
MODEL_SAVE_PATH = os.path.join(MODELS_DIR, "vit_best.pth")
TEMP_MODEL_PATH = os.path.join(MODELS_DIR, "temp_vit_model.pth")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# Hiperparâmetros e configuração do modelo
WEIGHTS = models.ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
IMG_SIZE = WEIGHTS.meta["min_size"][0]

BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 80

# Otimização e Early Stopping
LEARNING_RATE = 2e-5
HEAD_LR = 2e-4
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 0.05
GRAD_CLIP = 1.0
PATIENCE = 12
MIN_DELTA = 1e-4

# Amostragem, regularização e inferência
BALANCE_POWER = 1.0
EMA_DECAY = 0.995
USE_AMP = True
TTA = True

Treinando em: cuda


### Funções do Modelo

In [ ]:
# 1. Improved Huber Loss 
class ArticleHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, pred_logits, target):
        pred_probs = torch.softmax(pred_logits, dim=-1)
        target_one_hot = F.one_hot(target, num_classes=pred_logits.size(-1)).float()

        error1 = torch.abs(pred_probs - target_one_hot)

        quadratic = 0.5 * (error1 ** 2)
        linear = self.delta * error1 - 0.5 * (self.delta ** 2)

        loss = torch.where(error1 <= self.delta, quadratic, linear)
        return loss.mean()


# 2. Arquitetura Vision Transformer (ViT-B/16)
class ViTClassifier(nn.Module):
    def __init__(self, num_classes=3, dropout=0.3):
        super().__init__()
        self.vit = models.vit_b_16(weights=WEIGHTS)

        in_features = self.vit.heads.head.in_features
        self.vit.heads.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.vit(x)

### Divisão do Dataset (70/10/20)

In [3]:
# Estatísticas de normalização da ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transformações e Data Augmentation para treino (padronizadas com a FusNet)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=20, translate=(0.06, 0.06), scale=(0.9, 1.2),
        interpolation=InterpolationMode.BILINEAR
    ),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transformações para validação e teste
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
eval_full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=eval_transform)
num_classes = len(eval_full_dataset.classes)

print(f"Classes encontradas: {eval_full_dataset.classes}")
print(f"Total de imagens no dataset: {len(eval_full_dataset)}")

targets = eval_full_dataset.targets
indices = list(range(len(eval_full_dataset)))

# 1. Separação de 20% para teste e 80% para treino + validação
temp_idx, test_idx, temp_targets, test_targets = train_test_split(
    indices, targets,
    test_size=0.20,
    stratify=targets,
    random_state=SEED
)

# 2. Separação de 12.5% dos dados restantes (10% do total) para validação
train_idx, val_idx = train_test_split(
    temp_idx,
    test_size=0.125,
    stratify=temp_targets,
    random_state=SEED
)

train_dataset = Subset(train_full_dataset, train_idx)
val_dataset = Subset(eval_full_dataset, val_idx)
test_dataset = Subset(eval_full_dataset, test_idx)

# Amostragem balanceada por classe no treino (padronizada com a FusNet)
train_targets = np.array(targets)[train_idx]
class_counts = np.bincount(train_targets, minlength=num_classes)
sample_weights = (1.0 / class_counts[train_targets]) ** BALANCE_POWER
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED)
)

# Configuração dos DataLoaders
pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=pin)

def _contagem(idx):
    c = np.bincount(np.array(targets)[idx], minlength=num_classes)
    return {eval_full_dataset.classes[i]: int(c[i]) for i in range(num_classes)}

expected_share = np.bincount(train_targets, weights=sample_weights, minlength=num_classes)
expected_share = expected_share / expected_share.sum()
expected_share_dict = {eval_full_dataset.classes[i]: round(float(expected_share[i]), 3) for i in range(num_classes)}

print(f"\n[DIVISÃO CONCLUÍDA]")
print(f"Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_dataset)}")
print(f"  Treino    por classe: {_contagem(train_idx)}")
print(f"  Validação por classe: {_contagem(val_idx)}")
print(f"  Teste     por classe: {_contagem(test_idx)}")
print(f"  Fração esperada por classe em cada época de treino (amostrador): {expected_share_dict}")

Classes encontradas: ['healthy', 'red_spider_mite', 'rust']
Total de imagens no dataset: 1426

[DIVISÃO CONCLUÍDA]
Treino: 997 | Validação: 143 | Teste: 286
  Treino    por classe: {'healthy': 507, 'red_spider_mite': 109, 'rust': 381}
  Validação por classe: {'healthy': 73, 'red_spider_mite': 15, 'rust': 55}
  Teste     por classe: {'healthy': 146, 'red_spider_mite': 31, 'rust': 109}
  Fração esperada por classe em cada época de treino (amostrador): {'healthy': 0.333, 'red_spider_mite': 0.333, 'rust': 0.333}


### Treinamento

In [4]:
# Instancia o modelo ViT-B/16
model = ViTClassifier(num_classes=num_classes, dropout=0.3).to(device)

In [5]:
# Função de perda Improved Huber Loss (delta = 1.0)
criterion = ArticleHuberLoss(delta=1.0).to(device)

# Separação de parâmetros com e sem weight decay
def split_decay_params(named_params):
    decay, no_decay = [], []
    for name, p in named_params:
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or "class_token" in name or "pos_embedding" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return decay, no_decay

# Otimizador AdamW com taxas de aprendizado diferenciais (backbone e cabeça)
all_named = list(model.named_parameters())
bb_decay, bb_no_decay = split_decay_params([(n, p) for n, p in all_named if "heads" not in n])
hd_decay, hd_no_decay = split_decay_params([(n, p) for n, p in all_named if "heads" in n])

optimizer = torch.optim.AdamW([
    {'params': bb_decay,    'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY},
    {'params': bb_no_decay, 'lr': LEARNING_RATE, 'weight_decay': 0.0},
    {'params': hd_decay,    'lr': HEAD_LR,       'weight_decay': 1e-2},
    {'params': hd_no_decay, 'lr': HEAD_LR,       'weight_decay': 0.0},
])
assert sum(len(g['params']) for g in optimizer.param_groups) == len(all_named)

# Agendador de taxa de aprendizado: warmup linear seguido de decaimento cosseno
ETA_MIN_FACTOR = 0.02

def lr_factor(epoch):
    if epoch < WARMUP_EPOCHS:
        return 0.1 + 0.9 * epoch / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return ETA_MIN_FACTOR + (1.0 - ETA_MIN_FACTOR) * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)

# Média móvel exponencial (EMA) dos pesos
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY))

# Configuração de precisão mista (bfloat16)
USE_AMP = USE_AMP and device.type == 'cuda' and torch.cuda.get_device_capability()[0] >= 8

def amp_context():
    return torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_AMP)

print(f"Resolução de entrada: {IMG_SIZE}px | Mixed precision bf16: {USE_AMP}")

Resolução de entrada: 384px | Mixed precision bf16: True


In [6]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_bal_acc': [], 'lr': []}
best_bal_acc = -1.0
best_val_loss = float('inf')
best_epoch = 0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss, train_correct, n_seen = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            outputs = model(images)
        outputs = outputs.float()
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        ema.update_parameters(model)

        bs = labels.size(0)
        running_loss += loss.item() * bs
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        n_seen += bs

    epoch_train_loss = running_loss / n_seen
    epoch_train_acc = train_correct / n_seen

    # Validação com os pesos EMA
    val_model = ema.module
    val_model.eval()
    val_loss_sum, val_correct, n_val = 0.0, 0, 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with amp_context():
                outputs = val_model(images)
            outputs = outputs.float()
            loss = criterion(outputs, labels)

            bs = labels.size(0)
            val_loss_sum += loss.item() * bs
            n_val += bs
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss_sum / n_val
    epoch_val_acc = val_correct / n_val
    epoch_val_bal_acc = float(balanced_accuracy_score(val_labels, val_preds))

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_bal_acc'].append(epoch_val_bal_acc)
    history['lr'].append(current_lr)

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | "
        f"Val BalAcc: {epoch_val_bal_acc:.4f} | LR: {current_lr:.2e} | "
    )

    # Seleção do melhor modelo por Balanced Accuracy (desempate por Val Loss)
    improved = (epoch_val_bal_acc > best_bal_acc) or (
        epoch_val_bal_acc == best_bal_acc and epoch_val_loss < best_val_loss - MIN_DELTA
    )

    if improved:
        best_bal_acc = epoch_val_bal_acc
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save(val_model.state_dict(), TEMP_MODEL_PATH)
        print(f"  -> Melhoria! Val BalAcc: {best_bal_acc:.4f} | Val Loss: {best_val_loss:.4f} (Patience: 0/{PATIENCE})")
    else:
        epochs_no_improve += 1
        print(f"  -> Sem melhoria. (Patience: {epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"\n[EARLY STOPPING ATIVADO] Sem melhoria nas últimas {PATIENCE} épocas.")
            break

print(f"\nMelhor época: {best_epoch + 1} | Val BalAcc: {best_bal_acc:.4f} | Val Loss: {best_val_loss:.4f}")

if os.path.exists(TEMP_MODEL_PATH):
    shutil.move(TEMP_MODEL_PATH, MODEL_SAVE_PATH)
    print(f"\n[SUCESSO] Melhor modelo (pesos EMA) salvo em: {MODEL_SAVE_PATH}")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))

Epoch [01/80] Train Loss: 0.0764 | Train Acc: 0.6683 | Val Loss: 0.1230 | Val Acc: 0.3566 | Val BalAcc: 0.4854 | LR: 2.00e-06 | 
  -> Melhoria! Val BalAcc: 0.4854 | Val Loss: 0.1230 (Patience: 0/12)
Epoch [02/80] Train Loss: 0.0401 | Train Acc: 0.8427 | Val Loss: 0.0657 | Val Acc: 0.7692 | Val BalAcc: 0.7961 | LR: 8.00e-06 | 
  -> Melhoria! Val BalAcc: 0.7961 | Val Loss: 0.0657 (Patience: 0/12)
Epoch [03/80] Train Loss: 0.0381 | Train Acc: 0.8548 | Val Loss: 0.0290 | Val Acc: 0.9301 | Val BalAcc: 0.9086 | LR: 1.40e-05 | 
  -> Melhoria! Val BalAcc: 0.9086 | Val Loss: 0.0290 (Patience: 0/12)
Epoch [04/80] Train Loss: 0.0496 | Train Acc: 0.8286 | Val Loss: 0.0244 | Val Acc: 0.9091 | Val BalAcc: 0.9065 | LR: 2.00e-05 | 
  -> Sem melhoria. (Patience: 1/12)
Epoch [05/80] Train Loss: 0.0426 | Train Acc: 0.8478 | Val Loss: 0.0247 | Val Acc: 0.9091 | Val BalAcc: 0.9080 | LR: 2.00e-05 | 
  -> Sem melhoria. (Patience: 2/12)
Epoch [06/80] Train Loss: 0.0334 | Train Acc: 0.8770 | Val Loss: 0.0247 |

### Gráficos

In [7]:
# Gráficos de histórico de treinamento
epochs = range(1, len(history['train_loss']) + 1)
train_loss = history['train_loss']
val_loss = history['val_loss']
val_acc = history['val_acc']
train_acc = history.get('train_acc', None)

# 1. Acurácia por época
plt.figure(figsize=(6, 5))
if train_acc is not None and len(train_acc) > 0:
    plt.plot(epochs, train_acc, color='tab:blue', label='Training accuracy', linewidth=1.5)
plt.plot(epochs, val_acc, color='tab:orange', label='Validation accuracy', linewidth=1.5)
if 'val_bal_acc' in history and len(history['val_bal_acc']) > 0:
    plt.plot(epochs, history['val_bal_acc'], color='tab:green', linestyle=':', label='Validation balanced accuracy', linewidth=1.5)
plt.axvline(best_epoch + 1, color='gray', linestyle='--', linewidth=1.0, label=f'Best epoch ({best_epoch + 1})')
plt.xlabel('Epoch', fontweight='bold', fontsize=12)
plt.ylabel('Accuracy', fontweight='bold', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.savefig(os.path.join(RUNS_DIR, "accuracy_vs_epoch_ViT.png"), dpi=300, bbox_inches='tight')
plt.close()

# 2. Loss por época
plt.figure(figsize=(6, 5))
plt.plot(epochs, train_loss, color='tab:blue', label='Training loss', linewidth=1.5)
plt.plot(epochs, val_loss, color='tab:orange', label='Validation loss', linewidth=1.5)
plt.axvline(best_epoch + 1, color='gray', linestyle='--', linewidth=1.0, label=f'Best epoch ({best_epoch + 1})')
plt.xlabel('Epoch', fontweight='bold', fontsize=12)
plt.ylabel('Loss', fontweight='bold', fontsize=12)
plt.legend(loc='upper right', fontsize=11)
plt.savefig(os.path.join(RUNS_DIR, "loss_vs_epoch_ViT.png"), dpi=300, bbox_inches='tight')
plt.close()

# Inferência no conjunto de teste (TTA com 4 visões padronizado com a FusNet)
@torch.no_grad()
def predict_probs(net, loader, use_tta=False):
    net.eval()
    probs_all, labels_all = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with amp_context():
            probs = torch.softmax(net(images).float(), dim=1)
            if use_tta:
                views = [
                    torch.flip(images, dims=[3]),
                    torch.flip(images, dims=[2]),
                    torch.flip(images, dims=[2, 3]),
                ]
                for v in views:
                    probs = probs + torch.softmax(net(v).float(), dim=1)
                probs = probs / (1 + len(views))
        probs_all.append(probs.cpu().numpy())
        labels_all.append(labels.numpy())
    return np.concatenate(probs_all), np.concatenate(labels_all)

probs_plain, all_labels = predict_probs(model, test_loader, use_tta=False)
probs_tta, _ = predict_probs(model, test_loader, use_tta=True)

def _resumo_rapido(nome, probs):
    preds = probs.argmax(axis=1)
    return f"{nome}: Acc {(preds == all_labels).mean():.4f} | BalAcc {balanced_accuracy_score(all_labels, preds):.4f}"

print(_resumo_rapido("Teste sem TTA", probs_plain))
print(_resumo_rapido("Teste com TTA", probs_tta))
print(f"[O relatório abaixo usa TTA = {TTA}]\n")

all_probs = probs_tta if TTA else probs_plain
all_preds = all_probs.argmax(axis=1)

class_names = eval_full_dataset.classes
n_classes = len(class_names)

# 3. Matriz de Confusão
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 16}, cbar_kws={'shrink': 0.8})

plt.xlabel('Predicted Labels', fontweight='bold', fontsize=12)
plt.ylabel('True Labels', fontweight='bold', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.savefig(os.path.join(RUNS_DIR, "confusion_matrix_ViT.png"), dpi=300, bbox_inches='tight')
plt.close()

# Binarização dos rótulos para curvas multiclasse
y_test_bin = label_binarize(all_labels, classes=range(n_classes))
colors = sns.color_palette("husl", n_classes)

# 4. Curva ROC
plt.figure(figsize=(6, 5))
roc_auc_dict = {}

if n_classes == 2:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color='tab:orange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
else:
    for i, color in zip(range(n_classes), colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        roc_auc_dict[class_names[i]] = roc_auc
        plt.plot(fpr, tpr, color=color, lw=1.5, label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color='gray')
plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
plt.legend(loc="lower right", fontsize=10)
plt.savefig(os.path.join(RUNS_DIR, "roc_curve_ViT.png"), dpi=300, bbox_inches='tight')
plt.close()

# 5. Curva Precision-Recall
plt.figure(figsize=(6, 5))
pr_auc_dict = {}

if n_classes > 2:
    for i, color in zip(range(n_classes), colors):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, i], all_probs[:, i])
        ap_score = average_precision_score(y_test_bin[:, i], all_probs[:, i])
        pr_auc_dict[class_names[i]] = ap_score
        plt.plot(recall, precision, color=color, lw=1.5, label=f'{class_names[i]} (AP = {ap_score:.4f})')

    plt.xlabel('Recall (Sensibilidade)', fontweight='bold', fontsize=12)
    plt.ylabel('Precision (Precisão)', fontweight='bold', fontsize=12)
    plt.legend(loc="lower left", fontsize=10)
    plt.savefig(os.path.join(RUNS_DIR, "pr_curve_ViT.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 6. Relatório de métricas no conjunto de teste
bal_acc = balanced_accuracy_score(all_labels, all_preds)
report_str = classification_report(all_labels, all_preds, target_names=class_names, digits=4)

specificities = []
for i in range(n_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificities.append(spec)

macro_specificity = np.mean(specificities)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)

resumo_metricas = (
    "============================================================\n"
    "          AVALIAÇÃO REAL NO CONJUNTO DE TESTE (ViT)         \n"
    "============================================================\n"
    f"Acurácia Global (Padrão):     {(all_preds == all_labels).mean():.4f}\n"
    f"Acurácia Balanceada (Real):   {bal_acc:.4f}\n"
    f"Especificidade Média (Macro): {macro_specificity:.4f}\n"
    f"Precisão / Recall / F1 (Macro): {prec_m:.4f} / {rec_m:.4f} / {f1_m:.4f}\n\n"
    "--- Relatório Detalhado por Classe ---\n"
    f"{report_str}\n"
    "--- Especificidade, ROC-AUC e PR-AUC por Classe ---\n"
)

for idx, cls_name in enumerate(class_names):
    r_auc = roc_auc_dict.get(cls_name, 0.0)
    p_auc = pr_auc_dict.get(cls_name, 0.0)
    spec_cls = specificities[idx]
    resumo_metricas += (
        f" - {cls_name:<15} | Spec: {spec_cls:.4f} | "
        f"ROC-AUC: {r_auc:.4f} | PR-AUC (AP): {p_auc:.4f}\n"
    )

resumo_metricas += "============================================================\n"

print(resumo_metricas)

report_path = os.path.join(RUNS_DIR, "relatorio_metricas_ViT.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(resumo_metricas)

print(f"[SUCESSO] Gráficos e relatório de métricas salvos em: {RUNS_DIR}")

Teste sem TTA: Acc 0.8986 | BalAcc 0.8621
Teste com TTA: Acc 0.9091 | BalAcc 0.8867
[O relatório abaixo usa TTA = True]

          AVALIAÇÃO REAL NO CONJUNTO DE TESTE (ViT)         
Acurácia Global (Padrão):     0.9091
Acurácia Balanceada (Real):   0.8867
Especificidade Média (Macro): 0.9539
Precisão / Recall / F1 (Macro): 0.8589 / 0.8867 / 0.8704

--- Relatório Detalhado por Classe ---
                 precision    recall  f1-score   support

        healthy     0.9524    0.9589    0.9556       146
red_spider_mite     0.7027    0.8387    0.7647        31
           rust     0.9216    0.8624    0.8910       109

       accuracy                         0.9091       286
      macro avg     0.8589    0.8867    0.8704       286
   weighted avg     0.9136    0.9091    0.9103       286

--- Especificidade, ROC-AUC e PR-AUC por Classe ---
 - healthy         | Spec: 0.9500 | ROC-AUC: 0.9839 | PR-AUC (AP): 0.9853
 - red_spider_mite | Spec: 0.9569 | ROC-AUC: 0.9585 | PR-AUC (AP): 0.8362
 - rust 